# Per-Muscle and Overall Average Metrics — Augmented Dataset

For each algorithm that has a `gpu_lambda_column_compare_*_augmented.ipynb`,
this notebook reads the per-muscle-group result CSVs from
`{algo}/codes/results_augmented/`, computes the mean of every metric across
all 20 augmented volumes, appends an `Overall_Mean` row, and saves a summary
CSV. The final cell combines all `Overall_Mean` rows into one comparison
table with colour-coded Dice/Hausdorff columns and the best value per metric
bolded.

Unlike the Sheffield evaluation (37 fine-grained muscle labels), the
augmented dataset's ground truth only distinguishes 4 bilateral muscle
groups (myosegmenTUM `combined_gt` scheme): **Gracilis, Hamstrings,
Quadriceps, Sartorius**. Some algorithms (MedCLIP-SAMv2, MedCLIP-SAMv2
Text+Boxes, MedSegDiff) only predict Gracilis/Sartorius on this dataset, so
their rows for Hamstrings/Quadriceps are naturally absent.

In [ ]:
import pathlib
import re
import warnings
import numpy as np
import pandas as pd
from IPython.display import display

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
EVAL_DIR    = pathlib.Path(r'C:\Projects\dissector\eval_notebooks')
SUMMARY_DIR = EVAL_DIR / 'summary_results_augmented'
SUMMARY_DIR.mkdir(exist_ok=True)

# ── Augmented-dataset muscle groups (myosegmenTUM combined_gt scheme) ───────────
AUGMENTED_MUSCLES = ['gracilis', 'hamstrings', 'quadriceps', 'sartorius']

# Regex to strip '{muscle_name}_' prefix from column names
_AUG_PREFIX_RE = re.compile(
    r'^(' + '|'.join(re.escape(m) for m in sorted(AUGMENTED_MUSCLES, key=len, reverse=True)) + r')_',
    re.I,
)

# ── Canonical metric patterns (specific before general) ────────────────────────
_METRIC_RE = [
    ('inter_slice_dice_pred', re.compile(r'inter_slice_dice_pred',        re.I)),
    ('inter_slice_dice_gt',   re.compile(r'inter_slice_dice_gt',          re.I)),
    ('dice',                  re.compile(r'^(?:lower_)?dice$',            re.I)),
    ('hausdorff',             re.compile(r'hausdorff',                    re.I)),
    ('jaccard',               re.compile(r'jaccard',                      re.I)),
    ('volume_similarity',     re.compile(r'volume_similarity',            re.I)),
    ('false_negative',        re.compile(r'false.?neg|falseNeg',          re.I)),
    ('false_positive',        re.compile(r'false.?pos|falsePo',           re.I)),
    ('bce',                   re.compile(r'\bbce\b|binary_cross_entropy', re.I)),
    ('boundary_iou_3d',       re.compile(r'boundary_iou',                 re.I)),
]


def canonical_metric(col: str):
    # Strip muscle-name prefix then match to a canonical metric name.
    bare = _AUG_PREFIX_RE.sub('', col).rstrip(':')
    for name, pat in _METRIC_RE:
        if pat.search(bare):
            return name
    return None


def extract_muscle(stem: str):
    # Parse muscle name from df_{muscle}_{algo_tag}_augmented filename stem.
    s = stem.lower()
    if s.startswith('df_'):
        s = s[3:]
    if s.endswith('_augmented'):
        s = s[:-10]
    for muscle in sorted(AUGMENTED_MUSCLES, key=len, reverse=True):
        if s.startswith(muscle + '_') or s == muscle:
            return muscle
    return None


print('Helpers defined.')
print('Muscle prefix regex:', _AUG_PREFIX_RE.pattern[:80], '...')

In [ ]:
_FLOAT64_MAX = np.finfo(np.float64).max


def process_algorithm(label: str, results_dir: pathlib.Path):
    # Return a summary DataFrame for one algorithm, or None if no CSVs found.
    csv_files = sorted(results_dir.glob('df_*.csv'))
    if not csv_files:
        print(f'  [skip] no CSVs in {results_dir}')
        return None

    rows = []
    for csv_path in csv_files:
        muscle = extract_muscle(csv_path.stem)
        if muscle is None:
            print(f'  [skip] cannot identify muscle in {csv_path.name}')
            continue

        df = pd.read_csv(csv_path)
        metric_vals = {}
        for col in df.columns:
            metric_name = canonical_metric(col)
            if metric_name is None:
                continue
            s = pd.to_numeric(df[col], errors='coerce').to_numpy(dtype=np.float64)
            # Replace inf and extreme sentinel values with NaN
            s = np.where(np.isfinite(s) & (np.abs(s) < _FLOAT64_MAX), s, np.nan)
            if not np.isfinite(s).any():
                print(f'  [all non-finite] {csv_path.name} | {col}')
                continue
            metric_vals[metric_name] = np.nanmean(s, dtype=np.float64)

        # Derived: inter-slice dice ratio (pred / gt)
        pred = metric_vals.get('inter_slice_dice_pred')
        gt   = metric_vals.get('inter_slice_dice_gt')
        if pred is not None and gt is not None and gt > 0:
            metric_vals['inter_slice_dice_ratio'] = pred / gt

        row = {'muscle': muscle}
        row.update(metric_vals)
        rows.append(row)

    if not rows:
        return None

    summary = pd.DataFrame(rows).set_index('muscle')
    summary.insert(0, 'algorithm', label)

    numeric  = summary.select_dtypes(include='number').astype(np.float64)
    overall  = numeric.mean().rename('Overall_Mean')
    overall['algorithm'] = label
    summary  = pd.concat([summary, overall.to_frame().T])
    summary.index.name = 'muscle'
    return summary


print('process_algorithm defined.')

In [ ]:
# ── Algorithm registry ─────────────────────────────────────────────────────────
# Each entry: (display_label, relative_path_to_results_augmented_dir)
REGISTRY = [
    ('Dafne',                        'dafne/codes/results_augmented'),
    ('MuscleMap Thigh',              'muscle_map_thigh/codes/results_augmented'),
    ('MuscleMap WB',                 'muscle_map_wb/codes/results_augmented'),
    ('Hirriririir',                  'multimodal-multiethnic/codes/results_augmented'),
    ('MuSeg',                        'museg/codes/results_augmented'),
    ('MedCLIP-SAMv2',                'medclipsamv2/codes/results_augmented'),
    ('MedCLIP-SAMv2 Text+Boxes',     'medclipsamv2textboxes/augmented_results'),
    ('MedSegDiff',                   'medsegdiff/codes/results_augmented'),
]

print(f'{len(REGISTRY)} algorithms in registry:\n')
for label, rdir in REGISTRY:
    path = EVAL_DIR / rdir
    if path.exists():
        n = len(list(path.glob('df_*.csv')))
        status = f'{n} CSVs'
    else:
        status = 'DIR MISSING'
    print(f'  {label:<35}  {status}')

In [ ]:
# ── Process all algorithms ─────────────────────────────────────────────────────
summaries = {}  # label -> DataFrame

for label, rdir in REGISTRY:
    results_dir = EVAL_DIR / rdir
    print(f'\n── {label} ──')
    df = process_algorithm(label, results_dir)
    if df is None:
        continue
    summaries[label] = df

    num_cols = df.select_dtypes(include='number').columns.tolist()
    display(
        df.reset_index()
        .style
        .format('{:.4f}', subset=num_cols)
        .hide(axis='index')
    )

    safe_name = re.sub(r'[^\w]+', '_', label).strip('_').lower()
    out_path  = SUMMARY_DIR / f'{safe_name}_augmented_avg_metrics.csv'
    df.to_csv(out_path, float_format='%.4f')
    print(f'  Saved -> {out_path}')

print(f'\nProcessed {len(summaries)}/{len(REGISTRY)} algorithms.')

In [ ]:
# ── Combined Overall Means ─────────────────────────────────────────────────────
overall_rows = [
    df.loc[['Overall_Mean']]
    for df in summaries.values()
    if 'Overall_Mean' in df.index
]

if not overall_rows:
    print('No results yet — run the gpu_lambda_column_compare_*_augmented notebooks first.')
else:
    combined = pd.concat(overall_rows)
    combined.index = [row['algorithm'] for _, row in combined.iterrows()]
    combined.index.name = 'algorithm'
    combined = combined.drop(columns='algorithm')
    combined = combined.drop(
        columns=['inter_slice_dice_pred', 'inter_slice_dice_gt'], errors='ignore'
    )

    # Prefer dice as the primary sort key
    if 'dice' in combined.columns:
        combined = combined.sort_values('dice', ascending=False)

    num_cols = combined.select_dtypes(include='number').columns.tolist()
    grad_cols = {col: 'RdYlGn' for col in ['dice', 'jaccard', 'boundary_iou_3d',
                                             'inter_slice_dice_ratio']
                 if col in num_cols}
    grad_cols_r = {col: 'RdYlGn_r' for col in ['hausdorff', 'false_negative',
                                                  'false_positive', 'bce']
                   if col in num_cols}

    def _bold_best(s: pd.Series, higher_is_better: bool) -> list[str]:
        # Bold the best (max or min) value in a column; ties all get bolded.
        best = s.max() if higher_is_better else s.min()
        return ['font-weight: bold' if v == best else '' for v in s]

    styler = (
        combined.reset_index()
        .style
        .format('{:.4f}', subset=num_cols)
        .hide(axis='index')
    )
    for col, cmap in {**grad_cols, **grad_cols_r}.items():
        styler = styler.background_gradient(subset=[col], cmap=cmap, axis=0)
    for col in grad_cols:      # higher is better
        styler = styler.apply(_bold_best, subset=[col], higher_is_better=True)
    for col in grad_cols_r:    # lower is better
        styler = styler.apply(_bold_best, subset=[col], higher_is_better=False)

    display(styler)

    out_combined = SUMMARY_DIR / 'overall_means_augmented.csv'
    combined.to_csv(out_combined, float_format='%.4f')
    print(f'Saved -> {out_combined}')